In [13]:
# Cell 1: Environment Setup
!pip install pandas numpy scikit-learn matplotlib pyarrow -q

import pandas as pd
import numpy as np
import os, json, csv, time, random, logging
from datetime import date, datetime, timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: Python Fundamentals and Pandas")

✓ Environment ready for: Python Fundamentals and Pandas



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
# Cell 2: Generate Dataset
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✓ Dataset loaded: 10000 rows, 9 columns


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor,Laptop,North,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor,Monitor,North,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse,Keyboard,North,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub,Mouse,South,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard,Headset,West,22,285.08,2024-04-20,refunded,6271.76


In [15]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: Python Fundamentals and Pandas')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: Python Fundamentals and Pandas

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
  categ

In [16]:
# Cell 4: Core Implementation — Python Fundamentals and Pandas

import pandas as pd
import numpy as np

# ============================================================
# Data Cleaning Pipeline — a standard sequence for any dataset
# ============================================================

# Work on a copy so the original df is preserved for validation
df_work = df.copy()

# --- Step 1: Standardize text columns ---
# In real data, regions might come as 'north', 'NORTH', 'North'
df_work['region'] = df_work['region'].str.strip().str.title()
df_work['product'] = df_work['product'].str.strip().str.title()
df_work['status'] = df_work['status'].str.strip().str.lower()
print('=== 1. Text Standardization ===')
print(f'Unique regions: {sorted(df_work["region"].unique())}')

# --- Step 2: Inject and handle missing values (simulating real-world) ---
# Introduce some NaN values for learning purposes
np.random.seed(42)
mask = np.random.random(len(df_work)) < 0.02  # ~2% nulls
df_work.loc[mask, 'quantity'] = np.nan
print(f'\n=== 2. Missing Value Handling ===')
print(f'Nulls before fill: {df_work["quantity"].isnull().sum()}')

# Fill missing quantity with median (robust to outliers)
median_qty = df_work['quantity'].median()
df_work['quantity'] = df_work['quantity'].fillna(median_qty)
print(f'Nulls after fill:  {df_work["quantity"].isnull().sum()}')
print(f'Fill value used (median): {median_qty}')

# --- Step 3: Create derived columns ---
df_work['order_date'] = pd.to_datetime(df_work['order_date'])
df_work['order_month'] = df_work['order_date'].dt.to_period('M').astype(str)
df_work['order_quarter'] = df_work['order_date'].dt.quarter
df_work['day_of_week'] = df_work['order_date'].dt.day_name()
df_work['revenue'] = (df_work['quantity'] * df_work['unit_price']).round(2)
df_work['price_tier'] = pd.cut(df_work['unit_price'],
                                bins=[0, 50, 200, 500, 1000],
                                labels=['Budget', 'Mid', 'Premium', 'Luxury'])
print(f'\n=== 3. Derived Columns ===')
print(f'New columns: order_month, order_quarter, day_of_week, price_tier')
print(df_work[['order_id', 'order_month', 'order_quarter', 'day_of_week', 'price_tier']].head())

# --- Step 4: Filter / remove bad rows ---
df_clean = df_work[df_work['status'] != 'cancelled'].copy()
print(f'\n=== 4. Row Filtering ===')
print(f'Cleaned: {len(df_clean)} rows | Removed: {len(df_work) - len(df_clean)} cancelled rows')

# --- Step 5: Aggregation — equivalent to SQL GROUP BY ---
print('\n=== 5. Aggregation (GROUP BY equivalent) ===')

# Revenue by region
region_agg = df_clean.groupby('region').agg(
    order_count=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    avg_quantity=('quantity', 'mean')
).round(2).sort_values('total_revenue', ascending=False)
print('\nRevenue by Region:')
print(region_agg)

# Revenue by product and region (pivot table)
pivot = df_clean.pivot_table(
    values='revenue', index='product', columns='region',
    aggfunc='sum', fill_value=0
).round(2)
print('\nPivot Table — Revenue by Product x Region:')
print(pivot)

# Monthly trend
monthly = df_clean.groupby('order_month').agg(
    orders=('order_id', 'count'),
    revenue=('revenue', 'sum')
).round(2)
print('\nMonthly Revenue Trend:')
print(monthly)

# --- Step 6: Type conversion and memory optimization ---
print('\n=== 6. Type Optimization ===')
mem_before = df_clean.memory_usage(deep=True).sum()
df_clean['region'] = df_clean['region'].astype('category')
df_clean['product'] = df_clean['product'].astype('category')
df_clean['status'] = df_clean['status'].astype('category')
mem_after = df_clean.memory_usage(deep=True).sum()
print(f'Memory: {mem_before/1024:.1f} KB -> {mem_after/1024:.1f} KB '
      f'({(1 - mem_after/mem_before)*100:.1f}% reduction)')

print(f'\n-- Python Fundamentals and Pandas implementation complete')
print(f'Final dataset: {len(df_clean)} rows, {len(df_clean.columns)} columns')

=== 1. Text Standardization ===
Unique regions: ['East', 'North', 'South', 'West']

=== 2. Missing Value Handling ===
Nulls before fill: 197
Nulls after fill:  0
Fill value used (median): 25.0

=== 3. Derived Columns ===
New columns: order_month, order_quarter, day_of_week, price_tier
    order_id order_month  order_quarter day_of_week price_tier
0  ORD-00000     2024-04              2   Wednesday    Premium
1  ORD-00001     2024-01              1     Tuesday    Premium
2  ORD-00002     2024-11              4    Thursday    Premium
3  ORD-00003     2024-01              1    Thursday     Luxury
4  ORD-00004     2024-04              2    Saturday    Premium

=== 4. Row Filtering ===
Cleaned: 9015 rows | Removed: 985 cancelled rows

=== 5. Aggregation (GROUP BY equivalent) ===

Revenue by Region:
        order_count  total_revenue  avg_revenue  avg_quantity
region                                                       
West           2273    29493832.56     12975.73         25.27
East     

In [17]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: Python Fundamentals and Pandas')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: Python Fundamentals and Pandas
Input rows: 10000
Processing complete

Revenue by Region:
region
West     32704052.11
East     31929463.06
South    30902956.65
North    30756088.37
Name: revenue, dtype: float64


In [18]:
# Cell 6: Self-Check — Python and Pandas
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — Python and Pandas')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')


SELF-CHECK — Python and Pandas
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.
